# SDM Extraction Pipeline Demo

End-to-end extraction of species distribution modeling (SDM) requirements from a research PDF.

The pipeline:
1. Parses the PDF into sections (Abstract, Methods, Results, etc.)
2. Extracts structured `SDMRequirements` using targeted sections
3. Scores evidence confidence per field
4. Validates constraints (metric ranges, species format, etc.)
5. Retries critical errors with targeted re-extraction
6. Cross-references the extraction against the paper with a separate eval model
7. Computes an overall quality score (pass / marginal / fail)

## Setup

In [ ]:
from lit_review import SDMExtractionAgent, AgentConfig

PDF_PATH = "../papers/elith2010-range-shifting.pdf"

config = AgentConfig(
    model="gpt-4",
    eval_model="gpt-4",
    temperature=0.2,
)
agent = SDMExtractionAgent(config)

## 1. Section Parsing

Before calling the LLM, the pipeline parses the PDF into academic sections. This lets it send targeted text (Methods + Results) instead of the full truncated paper.

In [ ]:
from lit_review.pdf import extract_text
from lit_review.sections import parse_sections

text = extract_text(PDF_PATH)
sections = parse_sections(text)

print(f"Total text length: {len(text):,} chars")
print(f"Sections detected: {list(sections.sections.keys())}")
for name, body in sections.sections.items():
    print(f"  {name}: {len(body):,} chars")

## 2. Run the Full Pipeline

This single call runs extraction, validation, retry, evaluation, and quality scoring.

In [ ]:
result = await agent.run_pipeline(PDF_PATH)

## 3. Extracted Requirements

The structured extraction, grouped by methodology section.

In [ ]:
req = result.requirements

print("=== Study ===")
print(f"  Title:    {req.study.title}")
print(f"  Species:  {req.study.species}")
print(f"  Extent:   {req.study.geographic_extent}")

print("\n=== Occurrence ===")
print(f"  Type:       {req.occurrence.occurrence_type}")
print(f"  Presences:  {req.occurrence.total_presences}")
print(f"  Absences:   {req.occurrence.total_absences}")
print(f"  Source:     {req.occurrence.data_source}")

print("\n=== Predictors ===")
print(f"  Variables:   {req.predictors.variables}")
print(f"  Source:      {req.predictors.data_source}")
print(f"  Resolution:  {req.predictors.spatial_resolution}")

In [ ]:
print("=== Models ===")
for i, m in enumerate(req.models):
    best = " *best*" if m.is_best else ""
    print(f"\n  [{i}] {m.algorithm}{best}")
    if m.software:
        print(f"      Software: {m.software}")
    if m.hyperparameters:
        print(f"      Params:   {m.hyperparameters}")
    for p in m.performance:
        std = f" +/- {p.std}" if p.std else ""
        ctx = f" ({p.context})" if p.context else ""
        print(f"      {p.metric} = {p.value}{std}{ctx}")

In [ ]:
print("=== Evaluation Protocol ===")
print(f"  CV strategy: {req.evaluation.cv_strategy}")
print(f"  Metrics:     {req.evaluation.metrics_used}")
print(f"  Threshold:   {req.evaluation.threshold_method}")

print("\n=== Results ===")
print(f"  Key predictors:    {req.results.key_predictors}")
print(f"  Var importance:    {req.results.variable_importance}")
print(f"  Current dist:      {req.results.current_distribution}")
for s in req.results.projected_scenarios:
    print(f"  Scenario:          {s.scenario_name} — {s.description}")
print(f"  Projected dist:    {req.results.projected_distribution}")

## 4. Quality Gate

The pipeline combines validation, evaluation, and confidence into a single score.

In [ ]:
q = result.quality
print(f"Quality score: {q.score:.2f}  |  Grade: {q.grade.upper()}")
if q.reasons:
    print("Reasons:")
    for r in q.reasons:
        print(f"  - {r}")
print(f"\nRetries performed: {result.retries_performed}")

## 5. Confidence Report

Per-section evidence quality, scored without an LLM call.

In [ ]:
c = result.confidence
print(f"High: {c.num_high}  |  Low: {c.num_low}  |  Total: {len(c.field_scores)}\n")
for fc in c.field_scores:
    icon = {"high": "+", "medium": "~", "low": "!"}[fc.confidence]
    print(f"  [{icon}] {fc.field_path:12s}  {fc.confidence:6s}  {fc.reason}")

## 6. Validation Report

Constraint checks on the extracted values (metric ranges, species format, etc.).

In [ ]:
v = result.validation
print(f"Valid: {v.is_valid}  |  Errors: {v.num_errors}  |  Warnings: {v.num_warnings}")
if v.violations:
    print()
    for viol in v.violations:
        print(f"  [{viol.severity}] {viol.field_path}: {viol.rule} (got: {viol.actual_value})")
else:
    print("No violations found.")

## 7. Evaluation (Cross-Reference Check)

The eval model independently verifies each extracted field against the paper.

In [ ]:
e = result.evaluation
print(f"Verified: {e.num_verified}  |  Inaccurate: {e.num_inaccurate}  |  Unverifiable: {e.num_unverifiable}")
print(f"Assessment: {e.overall_assessment}\n")

for fv in e.field_verifications:
    icon = {"verified": "V", "inaccurate": "X", "unverifiable": "?"}[fv.status]
    print(f"  [{icon}] {fv.field_path}")
    print(f"      Extracted: {fv.extracted_value[:80]}")
    if fv.evidence:
        print(f"      Evidence:  {fv.evidence[:80]}")
    print()

## 8. JSON Export

The full result serializes cleanly for downstream use.

In [ ]:
import json

print(json.dumps(result.requirements.model_dump(exclude_none=True), indent=2)[:2000])